In [1]:
# import os
# import shutil
# from sklearn.model_selection import train_test_split

# #data splitting and making class dirs
# output_base = "dataset-v2/dataset_classified_split"
# label_dir = 'dataset-v2/kaggle-good/archive/'
# image_dir = 'dataset-v2/kaggle-good/archive/'

# # Gather (image, class_id) pairs
# samples = []
# for img_name in os.listdir(image_dir):
#     if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
#         continue
#     label_name = os.path.splitext(img_name)[0] + ".txt"
#     label_path = os.path.join(label_dir, label_name)
#     img_path = os.path.join(image_dir, img_name)
#     if not os.path.exists(label_path):
#         continue
#     with open(label_path, "r") as f:
#         first_line = f.readline().strip()
#         if not first_line:
#             continue
#         class_id = first_line.split()[0]
#     samples.append((img_path, class_id))

# # Stratified split
# img_paths, class_ids = zip(*samples)
# train_imgs, test_imgs, train_labels, test_labels = train_test_split(
#     img_paths, class_ids, test_size=0.2, stratify=class_ids, random_state=42
# )
# train_imgs, val_imgs, train_labels, val_labels = train_test_split(
#     train_imgs, train_labels, test_size=0.2, stratify=train_labels, random_state=42
# )

# splits = [
#     ("train", train_imgs, train_labels),
#     ("val", val_imgs, val_labels),
#     ("test", test_imgs, test_labels),
# ]

# # Copy images into split/class folders
# for split_name, imgs, labels in splits:
#     for img_path, class_id in zip(imgs, labels):
#         split_dir = os.path.join(output_base, split_name, class_id)
#         os.makedirs(split_dir, exist_ok=True)
#         shutil.copy(img_path, os.path.join(split_dir, os.path.basename(img_path)))

In [2]:
# from collections import Counter

# print("Final Split Distribution:")
# for name, labels in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
#     counts = Counter(labels)
#     print(f"{name}: {dict(counts)}")

In [ ]:
# import tensorflow as tf
# from tensorflow import keras

# IMG_SIZE = (224, 224)
# BATCH_SIZE = 32

# #1.1 Data augmentation pipeline
# data_augmentation = keras.Sequential([
#     keras.layers.RandomFlip("horizontal"),
#     keras.layers.RandomRotation(0.1),
#     keras.layers.RandomZoom(0.1),
#     keras.layers.RandomBrightness(0.1),
#     keras.layers.RandomContrast(0.1)
# ])

# # Normalization layer
# normalization_layer = keras.layers.Rescaling(1./255)

# #1.2 load data
# train_ds = keras.utils.image_dataset_from_directory(
#     "dataset-v2/dataset_classified_split/train",
#     image_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     shuffle=True,
#     color_mode="rgb"
# )
# val_ds = keras.utils.image_dataset_from_directory(
#     "dataset-v2/dataset_classified_split/val",
#     image_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     color_mode="rgb"
# )
# test_ds = keras.utils.image_dataset_from_directory(
#     "dataset-v2/dataset_classified_split/test",
#     image_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     color_mode="rgb"
# )

# # Apply augmentation and normalization
# train_ds = train_ds.map(lambda x, y: (data_augmentation(normalization_layer(x)), y))
# val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
# test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

Found 780 files belonging to 3 classes.
Found 196 files belonging to 3 classes.
Found 245 files belonging to 3 classes.


In [ ]:
# # Compute class weights for imbalanced data
# from sklearn.utils.class_weight import compute_class_weight
# import numpy as np

# # Get unique class ids and their counts from your training labels
# unique_classes = np.unique(train_labels)
# class_weights = compute_class_weight(
#     class_weight='balanced',
#     classes=unique_classes,
#     y=train_labels
# )
# class_weight_dict = {int(cls): weight for cls, weight in zip(unique_classes, class_weights)}
# print("Class weights:", class_weight_dict)

NameError: name 'train_labels' is not defined

In [ ]:
# # # Build and train ResNet50 classifier with class weights and fine-tuning
# # from tensorflow.keras import layers
# # from tensorflow import keras

# # Model definition
# def build_resnet_classifier(input_shape=IMG_SIZE + (3,), num_classes=3):
#     base_model = keras.applications.ResNet50(
#         include_top=False,
#         weights="imagenet",
#         input_shape=input_shape,
#         pooling="avg"
#     )
#     base_model.trainable = False  # Fine-tune later if needed

#     inputs = keras.Input(shape=input_shape)
#     x = base_model(inputs, training=False)
#     x = layers.Dropout(0.3)(x)
#     x = layers.Dense(128, activation="relu")(x)
#     x = layers.Dropout(0.3)(x)
#     outputs = layers.Dense(num_classes, activation="softmax")(x)
#     model = keras.Model(inputs, outputs)
#     return model

# resnet_model = build_resnet_classifier()
# resnet_model.compile(
#     optimizer=keras.optimizers.Adam(),
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"]
# )
# resnet_model.summary()

# # Train with class weights
# epochs = 15
# history = resnet_model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=epochs,
#     class_weight=class_weight_dict
# )

# # # Fine-tuning (optional, for better accuracy)
# # # Unfreeze the base model and recompile with a lower learning rate
# # # resnet_model.layers[1].trainable = True  # Unfreeze base_model
# # # resnet_model.compile(
# # #     optimizer=keras.optimizers.Adam(1e-5),
# # #     loss="sparse_categorical_crossentropy",
# # #     metrics=["accuracy"]
# # # )
# # # history_finetune = resnet_model.fit(
# # #     train_ds,
# # #     validation_data=val_ds,
# # #     epochs=5,
# # #     class_weight=class_weight_dict
# # # )

In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (
    MultiHeadAttention, LayerNormalization,
    GlobalAveragePooling2D, Reshape, Dense, Input, Dropout, Flatten
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# --- DATA LOADING & AUGMENTATION ---
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),
    keras.layers.RandomBrightness(0.1),
    keras.layers.RandomContrast(0.1)
])

train_ds = keras.utils.image_dataset_from_directory(
    "dataset-v2/dataset_classified_split/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    color_mode="rgb"
)
val_ds = keras.utils.image_dataset_from_directory(
    "dataset-v2/dataset_classified_split/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    color_mode="rgb"
)
test_ds = keras.utils.image_dataset_from_directory(
    "dataset-v2/dataset_classified_split/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    color_mode="rgb"
)

# Apply augmentation + preprocessing for ResNet50
preprocess_input = keras.applications.resnet50.preprocess_input
train_ds = train_ds.map(lambda x, y: (data_augmentation(preprocess_input(x)), y))
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y))
test_ds = test_ds.map(lambda x, y: (preprocess_input(x), y))

# Prefetch for performance
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

# --- CLASS WEIGHTS ---
# Update with your actual counts
class_counts = [532, 488, 201]  # class 0, class 1, class 2
class_labels = np.array([0]*class_counts[0] + [1]*class_counts[1] + [2]*class_counts[2])
class_weights = compute_class_weight('balanced', classes=np.unique(class_labels), y=class_labels)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print("Class weights:", class_weight_dict)

# --- MODEL ---
inputs = Input(shape=(224, 224, 3))
resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
resnet.trainable = True
for layer in resnet.layers[:-10]:  # unfreeze last 10 layers
    layer.trainable = False

x = resnet(inputs)
x = GlobalAveragePooling2D()(x)
x = Reshape((1, -1))(x)
x = MultiHeadAttention(num_heads=4, key_dim=64)(x, x)
x = LayerNormalization()(x)
x = Flatten()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(3, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)

# --- COMPILE ---
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# --- CALLBACKS ---
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)

# --- TRAIN ---
EPOCHS = 30
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weight_dict
)

# --- EVALUATE ---
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")


Found 780 files belonging to 3 classes.
Found 196 files belonging to 3 classes.
Found 245 files belonging to 3 classes.
Class weights: {0: np.float64(0.7650375939849624), 1: np.float64(0.8340163934426229), 2: np.float64(2.0248756218905473)}


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ input_layer_2[0]… │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 2048)   │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 2048)   │  2,099,968 │ reshape[0][0],    │
│ (MultiHeadAttentio… │                   │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 1, 2048)   │      4,096 │ multi_head_atten… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 2048)      │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 2048)      │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │    262,272 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 3)         │        387 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 25,954,435 (99.01 MB)

 Trainable params: 6,832,387 (26.06 MB)

 Non-trainable params: 19,122,048 (72.94 MB)

Epoch 1/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 48s 2s/step - accuracy: 0.4013 - loss: 1.2822 - val_accuracy: 0.4643 - val_loss: 0.9670 - learning_rate: 1.0000e-05
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 37s 1s/step - accuracy: 0.5000 - loss: 1.0069 - val_accuracy: 0.5000 - val_loss: 0.8884 - learning_rate: 1.0000e-05
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 37s 1s/step - accuracy: 0.5192 - loss: 0.9690 - val_accuracy: 0.5663 - val_loss: 0.8319 - learning_rate: 1.0000e-05
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.5615 - loss: 0.8645 - val_accuracy: 0.5816 - val_loss: 0.7997 - learning_rate: 1.0000e-05
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.5744 - loss: 0.8062 - val_accuracy: 0.5765 - val_loss: 0.7897 - learning_rate: 1.0000e-05
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 39s 2s/step - accuracy: 0.6115 - loss: 0.8194 - val_accuracy: 0.5918 - val_loss: 0.7806 - learning_rate: 1.0000e-05
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.6038 - loss:

In [6]:
# Unfreeze more layers (Stage 2)
for layer in resnet.layers[:-50]:  # previously :-10
    layer.trainable = False
for layer in resnet.layers[-50:]:
    layer.trainable = True

# Lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-6),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Continue training
history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,  # smaller number since we're fine-tuning deeper layers
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weight_dict
)

Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 64s 2s/step - accuracy: 0.6000 - loss: 0.7939 - val_accuracy: 0.5765 - val_loss: 0.9130 - learning_rate: 5.0000e-06
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6944 - loss: 0.6327
Epoch 2: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-06.
25/25 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - accuracy: 0.6641 - loss: 0.6775 - val_accuracy: 0.5969 - val_loss: 0.8743 - learning_rate: 5.0000e-06
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 52s 2s/step - accuracy: 0.7064 - loss: 0.6353 - val_accuracy: 0.5867 - val_loss: 0.8442 - learning_rate: 2.5000e-06
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6705 - loss: 0.6323
Epoch 4: ReduceLROnPlateau reducing learning rate to 1.249999968422344e-06.
25/25 ━━━━━━━━━━━━━━━━━━━━ 56s 2s/step - accuracy: 0.6846 - loss: 0.6297 - val_accuracy: 0.6020 - val_loss: 0.8089 - learning_rate: 2.5000e-06
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 55s 2s/step - accuracy: 0.6987 - loss: 0.6209 - va

In [ ]:
# # --- Stage 3: Full Unfreeze Fine-Tuning ---

# # 1. Unfreeze entire ResNet backbone
# for layer in resnet.layers:
#     layer.trainable = True

# # 2. Recompile with very small LR
# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=1e-6),
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"]
# )

# # 3. Train for just a few more epochs
# EPOCHS_STAGE3 = 5
# history_stage3 = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=EPOCHS_STAGE3,
#     callbacks=[early_stop, reduce_lr],
#     class_weight=class_weight_dict
# )


Epoch 1/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 168s 6s/step - accuracy: 0.6154 - loss: 0.7985 - val_accuracy: 0.5867 - val_loss: 0.8618 - learning_rate: 1.0000e-06
Epoch 2/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 136s 5s/step - accuracy: 0.6346 - loss: 0.7323 - val_accuracy: 0.5816 - val_loss: 0.8447 - learning_rate: 1.0000e-06
Epoch 3/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 136s 5s/step - accuracy: 0.6064 - loss: 0.7536 - val_accuracy: 0.5765 - val_loss: 0.8445 - learning_rate: 1.0000e-06
Epoch 4/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 136s 5s/step - accuracy: 0.6308 - loss: 0.7504 - val_accuracy: 0.5663 - val_loss: 0.8541 - learning_rate: 1.0000e-06
Epoch 5/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 137s 5s/step - accuracy: 0.6346 - loss: 0.7514 - val_accuracy: 0.5612 - val_loss: 0.8692 - learning_rate: 1.0000e-06
